## Observações

Importações identificadas como possivelmente não utilizadas:

- `from optbinning import OptimalBinning`
- `from sklearn.linear_model import LogisticRegression`
- `from sklearn.metrics import classification_report`
- `from sklearn.metrics import confusion_matrix`
- `from sklearn.model_selection import RandomizedSearchCV`
- `import lightgbm as lgb`
- `import matplotlib.pyplot as plt`
- `import pickle`
- `import seaborn as sns`
- `import shap`

### Descrição da célula

Esta célula executa uma etapa do fluxo de treinamento, avaliação ou preparação de dados do modelo XGBoost.

In [1]:
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
import shap
import joblib
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

from optbinning import OptimalBinning

d:\Ciencias de Dados\creditRisk_DecisionEngine\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Descrição da célula

Esta célula executa uma etapa do fluxo de treinamento, avaliação ou preparação de dados do modelo XGBoost.

In [2]:
import optuna
import xgboost as xgb
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score
)

### Descrição da célula

Esta célula executa uma etapa do fluxo de treinamento, avaliação ou preparação de dados do modelo XGBoost.

In [3]:
X_train = pd.read_parquet("../data/processed/X_train.parquet")
y_train = pd.read_parquet("../data/processed/y_train.parquet")

X_test = pd.read_parquet("../data/processed/X_test.parquet")
y_test = pd.read_parquet("../data/processed/y_test.parquet")

### Descrição da célula

Esta célula executa uma etapa do fluxo de treinamento, avaliação ou preparação de dados do modelo XGBoost.

In [4]:
cols_woes = []
cols_norm = []
for col in X_train.columns:
    if "_woe" in col:
        cols_woes.append(col)
    else:
        cols_norm.append(col)

## Model 

### Descrição da célula

Esta célula executa uma etapa do fluxo de treinamento, avaliação ou preparação de dados do modelo XGBoost.

In [5]:
def optimize_xgb(X, y, n_trials=30):

    def objective(trial):

        params = {
            "objective": "binary:logistic",
            "eval_metric": "auc",

            "n_estimators": trial.suggest_int(
                "n_estimators",
                100,
                1000
            ),

            "learning_rate": trial.suggest_float(
                "learning_rate",
                0.01,
                0.1,
                log=True
            ),

            "max_depth": trial.suggest_int(
                "max_depth",
                3,
                10
            ),

            "min_child_weight": trial.suggest_int(
                "min_child_weight",
                1,
                20
            ),

            "subsample": trial.suggest_float(
                "subsample",
                0.7,
                1.0
            ),

            "colsample_bytree": trial.suggest_float(
                "colsample_bytree",
                0.7,
                1.0
            ),

            "gamma": trial.suggest_float(
                "gamma",
                0,
                5
            ),

            "reg_alpha": trial.suggest_float(
                "reg_alpha",
                0,
                5
            ),

            "reg_lambda": trial.suggest_float(
                "reg_lambda",
                0,
                5
            ),

            "random_state": 42,
            "n_jobs": -1
        }

        model = xgb.XGBClassifier(**params)

        cv = StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=42
        )

        auc = cross_val_score(
            model,
            X,
            y,
            scoring="roc_auc",
            cv=cv,
            n_jobs=-1
        ).mean()

        return auc

    study = optuna.create_study(
        direction="maximize"
    )

    study.optimize(
        objective,
        n_trials=n_trials,
        show_progress_bar=True
    )

    return study.best_params

### Descrição da célula

Esta célula executa uma etapa do fluxo de treinamento, avaliação ou preparação de dados do modelo XGBoost.

In [6]:
best_params_norm = optimize_xgb(
    X_train[cols_norm],
    y_train,
    n_trials=30
)

xgb_norm = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1,
    **best_params_norm
)

xgb_norm.fit(
    X_train[cols_norm],
    y_train
)

[I 2026-06-09 23:09:43,080] A new study created in memory with name: no-name-ce056991-b088-44a1-b673-4b722249c416
Best trial: 0. Best value: 0.863482:   3%|▎         | 1/30 [00:04<02:16,  4.72s/it]

[I 2026-06-09 23:09:47,797] Trial 0 finished with value: 0.8634821142636582 and parameters: {'n_estimators': 250, 'learning_rate': 0.014123846940577929, 'max_depth': 4, 'min_child_weight': 5, 'subsample': 0.8458284202174792, 'colsample_bytree': 0.8463328956189886, 'gamma': 1.8711516855881678, 'reg_alpha': 1.3587797599828226, 'reg_lambda': 0.837637088582045}. Best is trial 0 with value: 0.8634821142636582.


Best trial: 1. Best value: 0.864681:   7%|▋         | 2/30 [00:09<02:17,  4.90s/it]

[I 2026-06-09 23:09:52,831] Trial 1 finished with value: 0.8646814391967051 and parameters: {'n_estimators': 376, 'learning_rate': 0.037742331542102346, 'max_depth': 6, 'min_child_weight': 1, 'subsample': 0.9325260038593455, 'colsample_bytree': 0.9016413779921146, 'gamma': 4.613863588706106, 'reg_alpha': 1.5285403899461532, 'reg_lambda': 0.20267652688833016}. Best is trial 1 with value: 0.8646814391967051.


Best trial: 2. Best value: 0.865153:  10%|█         | 3/30 [00:14<02:05,  4.66s/it]

...

Best trial: 10. Best value: 0.865682:  97%|█████████▋| 29/30 [02:26<00:05,  5.12s/it]

[I 2026-06-09 23:12:10,075] Trial 28 finished with value: 0.8656203673825008 and parameters: {'n_estimators': 952, 'learning_rate': 0.07614544235437427, 'max_depth': 3, 'min_child_weight': 15, 'subsample': 0.7212452227627255, 'colsample_bytree': 0.9785719663839791, 'gamma': 2.1324850242294775, 'reg_alpha': 4.127736245830628, 'reg_lambda': 1.9238357101931072}. Best is trial 10 with value: 0.8656823159115425.


Best trial: 10. Best value: 0.865682: 100%|██████████| 30/30 [02:33<00:00,  5.10s/it]


[I 2026-06-09 23:12:16,087] Trial 29 finished with value: 0.8639890897686339 and parameters: {'n_estimators': 941, 'learning_rate': 0.04605274342594841, 'max_depth': 4, 'min_child_weight': 14, 'subsample': 0.7186601818240071, 'colsample_bytree': 0.9430028588790669, 'gamma': 2.068149855424256, 'reg_alpha': 0.17651114219739705, 'reg_lambda': 1.0901853433762032}. Best is trial 10 with value: 0.8656823159115425.


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.9868041262130353
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import 

### Descrição da célula

Esta célula executa uma etapa do fluxo de treinamento, avaliação ou preparação de dados do modelo XGBoost.

In [7]:
best_params_woe = optimize_xgb(
    X_train[cols_woes],
    y_train,
    n_trials=30
)

xgb_woe = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1,
    **best_params_woe
)

xgb_woe.fit(
    X_train[cols_woes],
    y_train
)

[I 2026-06-09 23:12:17,773] A new study created in memory with name: no-name-224428b3-33eb-418f-b3ec-6c38204ec9b5
Best trial: 0. Best value: 0.863262:   3%|▎         | 1/30 [00:01<00:34,  1.19s/it]

[I 2026-06-09 23:12:18,961] Trial 0 finished with value: 0.8632618218043883 and parameters: {'n_estimators': 130, 'learning_rate': 0.04255103517489338, 'max_depth': 8, 'min_child_weight': 9, 'subsample': 0.7954916701645848, 'colsample_bytree': 0.7375248389024256, 'gamma': 2.849555424975256, 'reg_alpha': 1.007994821984085, 'reg_lambda': 1.5509459603109477}. Best is trial 0 with value: 0.8632618218043883.


Best trial: 0. Best value: 0.863262:   7%|▋         | 2/30 [00:06<01:38,  3.51s/it]

[I 2026-06-09 23:12:24,088] Trial 1 finished with value: 0.8628380753831362 and parameters: {'n_estimators': 792, 'learning_rate': 0.014523019172893326, 'max_depth': 4, 'min_child_weight': 20, 'subsample': 0.8818636812559263, 'colsample_bytree': 0.7199041412038301, 'gamma': 4.822619070755757, 'reg_alpha': 0.2825850516772149, 'reg_lambda': 2.5554008567119184}. Best is trial 0 with value: 0.8632618218043883.


Best trial: 0. Best value: 0.863262:  10%|█         | 3/30 [00:08<01:18,  2.92s/it]

...

Best trial: 18. Best value: 0.863974:  97%|█████████▋| 29/30 [02:22<00:06,  6.19s/it]

[I 2026-06-09 23:14:40,446] Trial 28 finished with value: 0.8635400283300445 and parameters: {'n_estimators': 786, 'learning_rate': 0.019762512344338137, 'max_depth': 9, 'min_child_weight': 16, 'subsample': 0.8148991797965863, 'colsample_bytree': 0.7836749507466154, 'gamma': 1.1097828259784936, 'reg_alpha': 4.266351578625125, 'reg_lambda': 2.3896733803542607}. Best is trial 18 with value: 0.8639742368796695.


Best trial: 18. Best value: 0.863974: 100%|██████████| 30/30 [02:27<00:00,  4.92s/it]


[I 2026-06-09 23:14:45,429] Trial 29 finished with value: 0.8634769086015837 and parameters: {'n_estimators': 744, 'learning_rate': 0.031564043249930736, 'max_depth': 9, 'min_child_weight': 18, 'subsample': 0.8190424696109415, 'colsample_bytree': 0.7814284453955735, 'gamma': 2.7859081703085624, 'reg_alpha': 1.7041829755262818, 'reg_lambda': 1.515581756967983}. Best is trial 18 with value: 0.8639742368796695.


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.7698257580702267
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import 

### Descrição da célula

Esta célula executa uma etapa do fluxo de treinamento, avaliação ou preparação de dados do modelo XGBoost.

In [8]:
def evaluate_model(
    model,
    X_test,
    y_test,
    model_name
):

    y_prob = model.predict_proba(X_test)[:, 1]

    y_pred = model.predict(X_test)

    auc = roc_auc_score(
        y_test,
        y_prob
    )

    gini = 2 * auc - 1

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    return {
        "model": model_name,
        "auc": round(auc, 4),
        "gini": round(gini, 4),
        "accuracy": round(accuracy, 4)
        
    }

### Descrição da célula

Esta célula executa uma etapa do fluxo de treinamento, avaliação ou preparação de dados do modelo XGBoost.

In [9]:
result_norm = evaluate_model(
    xgb_norm,
    X_test[cols_norm],
    y_test,
    "XGB_NORM"
)

result_woe = evaluate_model(
    xgb_woe,
    X_test[cols_woes],
    y_test,
    "XGB_WOE"
)

results = pd.DataFrame([
    result_norm,
    result_woe
])



In [10]:
joblib.dump(xgb_norm, "../assets/models/xgb_norm.joblib")
joblib.dump(xgb_woe, "../assets/models/xgb_woe.joblib")

['../assets/models/xgb_woe.joblib']

### Descrição da célula

Esta célula executa uma etapa do fluxo de treinamento, avaliação ou preparação de dados do modelo XGBoost.

In [11]:
results

,model,auc,gini,accuracy
0,XGB_NORM,0.8662,0.7325,0.9378
1,XGB_WOE,0.8639,0.7278,0.9363
